# **Text Summarization using Transformer Model**

This project implements an **abstractive text summarization model** using a transformer-based approach. The model takes a long article as input and generates a short, meaningful summary.

---

## **Model Used**

- **T5 (Text-to-Text Transfer Transformer)**
- Converts input text → output text (summarization)

**Why T5?**
- Generates human-like summaries  
- Understands context + creates new sentences  
- Lightweight model (`t5-small`) for fast training  

---

## **How the Model Works**

1. Input article is converted into tokens  
2. Transformer processes the text  
3. Model learns article → summary mapping  
4. Generates summary for new input  

---

## **Why T5 instead of BERT**

- **BERT** → only understands text  
- **T5** → understands + generates text  
> That’s why T5 is used

---

## **Model Source**

- Loaded from **Hugging Face Transformers**
- Pre-trained model: `t5-small`

---

## **Conclusion**

- Transformer-based model used for summarization  
- Generates short summaries from long text  
- Evaluated using ROUGE score  

## **1. Importing Required Libraries**

In this step, all necessary libraries are imported for data processing, model building, and training.

- **Pandas & NumPy** → used for handling and processing data  
- **Matplotlib** → used for visualization  
- **Transformers** → used to load tokenizer, model, and training utilities  
- **Datasets** → used to convert data into model-friendly format  
- **PyTorch** → backend for deep learning operations  
- **Scikit-learn** → used for data handling and evaluation  

This step ensures that all tools required for building the summarization model are available.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments
)
from datasets import Dataset
import torch
from sklearn.model_selection import train_test_split

print("All libraries imported successfully!")

All libraries imported successfully!


## **2. Loading and Understanding the Dataset**

In this step, the dataset is loaded using Pandas. The dataset used is the [**CNN/DailyMail News Summarization dataset**](https://www.kaggle.com/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail), which contains news articles and their corresponding summaries.

- **article** → input text (long news content)  
- **highlights** → target summary (short summary)

This dataset contains **300,000+ news articles** written by CNN and DailyMail journalists and is widely used for training text summarization models.

The dataset supports both:
- **Extractive summarization** (selecting sentences)  
- **Abstractive summarization** (generating new summary text)  


## **It include Train & Test Data**

- **Train dataset** → teaches the model  
- **Test dataset** → checks performance on new data  


> In this project, we use it for **abstractive summarization**, where the model generates a new summary instead of copying sentences.

The function `df.head()` is used to display the first few rows of the dataset to understand its structure and content.

In [ ]:
df = pd.read_csv("/content/train.csv", on_bad_lines="skip", engine="python")
df.head()

,id,article,highlights
0,0001d1afc246a7964130f43ae940af6bc6c57f01,By . Associated Press . PUBLISHED: . 14:11 EST...,"Bishop John Folda, of North Dakota, is taking ..."
1,0002095e55fcbd3a2f366d9bf92a95433dc305ef,(CNN) -- Ralph Mata was an internal affairs li...,Criminal complaint: Cop used his role to help ...
2,00027e965c8264c35cc1bc55556db388da82b07f,A drunk driver who killed a young woman in a h...,"Craig Eccleston-Todd, 27, had drunk at least t..."
3,0002c17436637c4fe1837c935c04de47adb18e9a,(CNN) -- With a breezy sweep of his pen Presid...,Nina dos Santos says Europe must be ready to a...
4,0003ad6ef0c37534f80b55b4235108024b407f0b,Fleetwood are the only team still to have a 10...,Fleetwood top of League One after 2-0 win at S...


## **3. Data Inspection and Analysis**

In this step, basic analysis of the dataset is performed to understand its structure and quality.

### **1. Dataset Shape**
- The dataset contains **144,209 rows and 3 columns**
- This indicates a large dataset suitable for training deep learning models

### **2. Column Names**
- **id** → unique identifier (not required for training)  
- **article** → input text (long news article)  
- **highlights** → target summary  

Only **article** and **highlights** are useful for this project

---

### **3. Missing Values**
- No missing values found in any column  
> Dataset is clean and ready for processing  

---

### **4. Article Length Analysis**
- Average length ≈ **4029 characters**
- Maximum length ≈ **15,925 characters**

 Articles are very long, which can slow down training and exceed model limits  

---

### **5. Summary Length Analysis**
- Average length ≈ **295 characters**
- Summaries are much shorter than articles  

 This confirms that the dataset is suitable for **text summarization tasks**

---

### **Conclusion**
- Dataset is clean (no missing values)  
- Articles are long and summaries are short  
- Suitable for training an **abstractive summarization model**  

In [ ]:
# Shape of data
print("Shape:", df.shape)

# Column names
print("\nColumns:", df.columns)

# Missing values
print("\nMissing Values:\n", df.isnull().sum())

# Length check
print("\nArticle length:\n", df["article"].str.len().describe())
print("\nSummary length:\n", df["highlights"].str.len().describe())

Shape: (144209, 3)

Columns: Index(['id', 'article', 'highlights'], dtype='object')

Missing Values:
 id            0
article       0
highlights    0
dtype: int64

Article length:
 count    144209.000000
mean       4029.884092
std        1955.227981
min          48.000000
25%        2578.000000
50%        3672.000000
75%        5120.000000
max       15925.000000
Name: article, dtype: float64

Summary length:
 count    144209.000000
mean        294.775319
std         119.352450
min          14.000000
25%         218.000000
50%         280.000000
75%         343.000000
max        7388.000000
Name: highlights, dtype: float64


## **4. Data Cleaning and Preprocessing**

In this step, the dataset is cleaned and prepared for training the model.

### 1. Selecting Required Columns
- Only **article** and **highlights** columns are selected  
- The **id** column is removed as it is not useful for training  

---

### 2. Renaming Columns
- **article → text**  
- **highlights → summary**  

This makes the dataset more readable and compatible with the model.

---

### 3. Removing Small Text
- Articles with length less than **100 characters** are removed  
- Summaries with length less than **20 characters** are removed  

This helps in removing incomplete or low-quality data.

---

### 4. Sampling the Dataset
- A subset of **10,000 rows** is selected  

Reason:
- The full dataset is very large  
- Reduces training time  
- Makes the model more efficient  

---

### In the end
- Unnecessary columns are removed  
- Data is cleaned and filtered  
- Dataset size is reduced for faster training  
- Data is now ready for model training  

In [ ]:
# Keep required columns
df = df[["article", "highlights"]]

# Rename
df.rename(columns={
    "article": "text",
    "highlights": "summary"
}, inplace=True)

# Remove very small text
df = df[df["text"].str.len() > 100]
df = df[df["summary"].str.len() > 20]

# Take subset (IMPORTANT)
df = df.sample(10000).reset_index(drop=True)

df.head()

,text,summary
0,By . Luke Augustus for MailOnline . Follow @@L...,Cristiano Ronaldo has won the UEFA Best Player...
1,London (CNN) -- If only Mitt Romney could turn...,Mitt Romney's comments in London draw criticis...
2,(CNN) -- English Premier League Arsenal signed...,Arsenal sign teenage winger Alex Oxlade-Chambe...
3,"By . Mark Duell . UPDATED: . 17:06 EST, 3 Marc...",Two mattress retailers bow to community pressu...
4,Cambodian police used cattle prods to stun wor...,"Around 3,000 mostly female workers blocked a r..."


## **5. Data Conversion and Tokenization**

In this step, the cleaned dataset is converted into a format suitable for training the transformer model, and tokenization is applied.

### 1. Converting DataFrame to Dataset
- The Pandas DataFrame is converted into a Hugging Face **Dataset** object  
- This format is required for efficient model training  

---

### 2. Loading Tokenizer
- The tokenizer for **t5-small** model is loaded  
- It converts text into numerical tokens that the model can understand  

---

### 3. Input Preparation
- Each input text is prefixed with:  
  **"summarize accurately: "**  

This helps the model understand the task (summarization).

---

### 4. Tokenizing Input Text
- Input text is converted into tokens  
- **max_length = 512** → limits article length  
- **truncation = True** → cuts long text  
- **padding = max_length** → ensures equal length  

---

### 5. Tokenizing Target Summary
- Summary is also tokenized  
- **max_length = 128** → summaries are shorter  

---

### 6. Creating Labels
- The tokenized summary is stored as **labels**  
- This is used by the model during training to learn correct outputs  

---

### 7. Applying Tokenization
- The function is applied to the entire dataset using **map()**  
- **batched = True** improves processing speed   

In [ ]:
dataset = Dataset.from_pandas(df)
tokenizer = AutoTokenizer.from_pretrained("t5-small")

def tokenize(example):
    inputs = ["summarize accurately: " + text for text in example["text"]]

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["summary"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

## **6. Model Loading and Training Setup**

In this step, the transformer model is loaded and configured for training.

### 1. Loading the Model
- The pre-trained **t5-small** model is loaded using Hugging Face  
- This model is already trained on large text data and is fine-tuned for summarization  

---

### 2. Training Arguments
The training configuration is defined using **TrainingArguments**:

- **output_dir** → directory to store results  
- **per_device_train_batch_size = 2** → number of samples processed at a time  
- **num_train_epochs = 2** → number of times the model will see the entire dataset  
- **logging_steps = 500** → prints training progress after every 500 steps  
- **save_strategy = "no"** → model checkpoints are not saved during training  
- **fp16 = True** → uses half-precision for faster training (if GPU available)  

---

### 3. Trainer Initialization
- The **Trainer** class is used to manage the training process  
- It combines:
  - model  
  - training arguments  
  - dataset  

This simplifies the training workflow.

In [ ]:
# Load model
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=2,
    logging_steps=500,
    save_strategy="no",
    fp16=True
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## **7. Model Training**

In this step, the training process of the model is started.

- The **trainer.train()** function is used to train the model on the prepared dataset  
- The model learns the relationship between **input text (article)** and **output text (summary)**  

During training:
- The model processes data in batches  
- It updates its internal parameters to minimize error  
- Loss values are calculated to measure how well the model is learning  

---

### Output
- Training logs are displayed at regular intervals  
- Loss decreases as the model improves  

In [ ]:
trainer.train()

Step,Training Loss
500,1.706322
1000,1.194649
1500,1.186939
2000,1.161542
2500,1.159959
3000,1.175484
3500,1.192931
4000,1.169516
4500,1.176625
5000,1.153760


TrainOutput(global_step=10000, training_loss=1.1806216857910157, metrics={'train_runtime': 1002.1128, 'train_samples_per_second': 19.958, 'train_steps_per_second': 9.979, 'total_flos': 2706836029440000.0, 'train_loss': 1.1806216857910157, 'epoch': 2.0})

## **8. Test Data Preparation**

In this step, the test dataset is loaded and prepared for evaluating the trained model.

### 1. Loading Test Dataset
- The **test.csv** file is loaded using Pandas  
- This dataset contains unseen data that was not used during training  

---

### 2. Selecting Required Columns
- Only **article** and **highlights** columns are selected  
- Other columns are removed as they are not required  

---

### 3. Renaming Columns
- **article → text**  
- **highlights → summary**  

This keeps consistency with the training dataset format.

---

### 4. Sampling the Data
- A small sample of **5 rows** is selected  

Reason:
- Quick testing  
- Reduces execution time  

---

### 5. Displaying Data
- `test_df.head()` is used to view sample test data  

In [ ]:
test_df = pd.read_csv("/content/test.csv", on_bad_lines="skip", engine="python")

# Same cleaning
test_df = test_df[["article", "highlights"]]
test_df.rename(columns={"article": "text", "highlights": "summary"}, inplace=True)

# Take small sample
test_df = test_df.sample(5)

test_df.head()

,text,summary
1516,Comedian Jenny Eclair travelled with her other...,The comedian stayed with Flavours who offer a ...
1393,A woman of Arab and Jewish descent who was str...,The federal government will give Shoshana Hebs...
10560,World No 1 Novak Djokovic has apologised to th...,Novak Djokovic beat Andy Murray 7-6 4-6 6-0 in...
11457,(CNN)ISIS on Wednesday released more than 200 ...,Most of those released were women and children...
647,Hillary Clinton’s security detail arrived at a...,"Second modified, armored van spotted near Des ..."


## **9. Summary Generation Function**

In this step, a function is created to generate summaries from input text using the trained model.

### 1. Device Selection
- Checks if GPU (**cuda**) is available, otherwise uses CPU  
- The model is moved to the selected device for faster computation  

---

### 2. Input Preparation
- Input text is prefixed with:  
  **"summarize important points: "**  

This guides the model to focus on key information.

- The text is tokenized using the tokenizer  
- **max_length = 512** limits input size  
- **truncation = True** removes extra long text  

---

### 3. Generating Summary
- The model generates summary using **generate()** function  

Key parameters:
- **max_length = 130** → controls maximum summary length  
- **min_length = 40** → ensures summary is not too short  
- **num_beams = 6** → uses beam search for better output quality  
- **repetition_penalty = 2.0** → reduces repeated words  
- **length_penalty = 1.2** → balances summary length  
- **early_stopping = True** → stops generation when complete  

---

### 4. Decoding Output
- Generated token IDs are converted back to text  
- Special tokens are removed  

In [ ]:
def summarize(text):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    inputs = tokenizer(
        "summarize important points: " + text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    summary_ids = model.generate(
    inputs["input_ids"],
    max_length=130,
    min_length=40,
    num_beams=6,
    repetition_penalty=2.0,
    length_penalty=1.2,
    early_stopping=True
)
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

## **10. User Input and Interactive Testing**

In this step, an interactive loop is created to allow the user to input custom text and generate summaries.

### 1. Infinite Loop
- A **while loop** is used to continuously take user input  
- This allows multiple summaries without restarting the program  

---

### 2. User Input
- The user enters any text using the **input()** function  
- The prompt also provides an option to exit  

---

### 3. Exit Condition
- If the user enters **"exit"**, **"stop"**, or **"quit"**, the loop breaks  
- This safely stops the program  

---

### 4. Generating Summary
- The entered text is passed to the **summarize()** function  
- The model generates and prints the summary  


In [ ]:
while True:
    text = input("\nEnter text (type 'exit' to stop): ")

    if text.lower() in ["exit", "stop", "quit"]:
        break

    print("\nSummary:")
    print(summarize(text))


Enter text (type 'exit' to stop): Lionel Andrés "Leo" Messi (born 24 June 1987) is an Argentine professional footballer who plays as a forward for and captains both the Major League Soccer club Inter Miami and the Argentina national team. Widely regarded as one of the greatest players in history, Messi has set numerous records for individual accolades won throughout his professional footballing career, including eight Ballon d'Ors, six European Golden Shoes, and eight times being named the world's best player by FIFA. In 2025, he was named the All Time Men's World Best Player by the IFFHS.  Messi is the most decorated player in the history of professional football, having won 46 team trophies.[note 3] His records include most goals in a calendar year (91), most goals for a single club (672 for Barcelona), most goals in La Liga (474), most assists in international football (61), most goal contributions in the FIFA World Cup (21), and most goal contributions in the Copa América (32). Me

## **11. Model Evaluation using ROUGE Score**

In this step, the performance of the trained model is evaluated using the **ROUGE metric**, which is widely used for text summarization tasks.

---

### What is ROUGE?

**ROUGE (Recall-Oriented Understudy for Gisting Evaluation)** is a metric used to measure the similarity between the **generated summary** and the **actual summary**.

It works by comparing overlapping words and phrases.

---

### Types of ROUGE Scores

- **ROUGE-1** → measures word-level overlap  
- **ROUGE-2** → measures 2-word phrase overlap  
- **ROUGE-L** → measures sentence structure similarity  

---

### Why ROUGE is Used

- Summarization is a **text generation task**  
- The same meaning can be written in different ways  
- Exact matching is not possible  

ROUGE helps in evaluating how much important information is shared between the predicted and actual summaries.

---

### Why Accuracy is Not Used

- Accuracy works for classification tasks (correct / incorrect output)  
- In summarization, there is **no single correct answer**  
- Different summaries can have the same meaning but different words  

Example:
- Actual: "Ronaldo is a great player"  
- Predicted: "Cristiano Ronaldo is one of the best footballers"  

Both are correct in meaning, but accuracy would treat them as different.

---

### Evaluation Process

- A sample of **50 rows** is selected  
- For each row:
  - Input text is taken  
  - Actual summary is stored  
  - Model generates predicted summary  

- Two lists are created:
  - **predictions** → generated summaries  
  - **references** → actual summaries  

---

### Computing ROUGE Score

- The **rouge.compute()** function compares predictions with references  
- It returns ROUGE-1, ROUGE-2, and ROUGE-L scores  


In [ ]:
import evaluate

rouge = evaluate.load("rouge")

# take small sample for speed
sample_df = df.sample(50)

predictions = []
references = []

for i in range(len(sample_df)):
    text = sample_df["text"].iloc[i]
    actual = sample_df["summary"].iloc[i]

    pred = summarize(text)

    predictions.append(pred)
    references.append(actual)

# compute score
scores = rouge.compute(predictions=predictions, references=references)

print(scores)

{'rouge1': np.float64(0.40112755516235543), 'rouge2': np.float64(0.20240766420920997), 'rougeL': np.float64(0.3124309763487164), 'rougeLsum': np.float64(0.36140738899024255)}
